In [3]:
import requests
import pandas as pd
from datetime import datetime

print("✅ Setup complete")

✅ Setup complete


In [4]:
def get_coordinates(city_name):
    try:
        url = f"https://nominatim.openstreetmap.org/search?city={city_name}&format=json&limit=1"
        response = requests.get(url, timeout=5, headers={'User-Agent': 'WeatherAgent/1.0'})
        data = response.json()

        if data:
            lat = float(data[0]['lat'])
            lon = float(data[0]['lon'])
            full_name = data[0].get('display_name', city_name)
            return lat, lon, full_name
        else:
            return None, None, None
    except Exception as e:
        print(f"Error: {e}")
        return None, None, None

print("✅ Coordinates function ready")

✅ Coordinates function ready


In [5]:
def get_weather(city_name):

    lat, lon, full_name = get_coordinates(city_name)

    if lat is None:
        return {"error": f"City '{city_name}' not found"}

    print(f"🌍 Fetching weather for {full_name}...")

    try:
        url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current=temperature_2m,relative_humidity_2m,weather_code,wind_speed_10m"

        response = requests.get(url, timeout=5)
        data = response.json()

        weather = {
            "city": full_name,
            "temperature": data['current']['temperature_2m'],
            "humidity": data['current']['relative_humidity_2m'],
            "wind_speed": data['current']['wind_speed_10m']
        }

        return weather

    except Exception as e:
        return {"error": str(e)}

print("✅ Weather function ready")

✅ Weather function ready


In [6]:
def get_air_quality(city_name):

    try:
        url = f"https://api.openaq.org/v1/latest?city={city_name}&limit=1"
        response = requests.get(url, timeout=5)
        data = response.json()

        if data['results']:
            result = data['results'][0]
            pm25 = result['measurements'][0]['value'] if result['measurements'] else "N/A"
            return {"pm25": pm25}
        else:
            return {"pm25": "N/A"}
    except:
        return {"pm25": "N/A"}

print("✅ Air Quality function ready")

✅ Air Quality function ready


In [7]:
def get_all_data(city_name):

    weather = get_weather(city_name)
    air = get_air_quality(city_name)

    if "error" in weather:
        return None

    combined = {
        "city": weather['city'],
        "temperature": weather['temperature'],
        "humidity": weather['humidity'],
        "wind_speed": weather['wind_speed'],
        "pm25": air['pm25']
    }

    return combined

print("✅ Combined function ready")

✅ Combined function ready


In [8]:
data = get_all_data("London")
print(data)

🌍 Fetching weather for Greater London, England, United Kingdom...
{'city': 'Greater London, England, United Kingdom', 'temperature': 19.4, 'humidity': 78, 'wind_speed': 16.9, 'pm25': 'N/A'}


In [9]:
cities = ["London", "Tokyo", "Lahore", "Paris"]
results = []

for city in cities:
    print(f"\n📍 {city}:")
    data = get_all_data(city)
    if data:
        results.append(data)

df = pd.DataFrame(results)
df.to_csv('weather_data.csv', index=False)
print("\n✅ Saved to: weather_data.csv")


📍 London:
🌍 Fetching weather for Greater London, England, United Kingdom...

📍 Tokyo:
🌍 Fetching weather for 東京都, 日本...

📍 Lahore:
🌍 Fetching weather for لاہور, تحصیل لاہور شہر, ضلع لاہور, لاہور ڈویژن, پنجاب, 54100, پاکستان...

📍 Paris:
🌍 Fetching weather for Paris, Île-de-France, France métropolitaine, France...

✅ Saved to: weather_data.csv
